# Natural Presenter AI — F5-TTS Fine-Tuning (University JupyterLab — NVIDIA L40 44GB)

Optimized for **Linux JupyterLab + NVIDIA L40 (44.4 GB VRAM, PyTorch 2.6+cu124)**.

### Quick Start (3 Steps)
1. In JupyterLab's left sidebar, create a folder (e.g., `natural-presenter-ai`) and click the **Upload (`↑`)** button to upload:
   - `vn_01_f5_dataset.zip` (~61 MB)
   - `05_finetune_f5tts_jupyter_l40.ipynb`
2. Open this notebook and run **Cells 1 → 5** in order.
3. Training ~27 minutes of audio takes **~15–25 minutes** on NVIDIA L40. Listen to the comparison in **Cell 5**, then right-click the best checkpoint inside `ckpts/vn_01/` in the left sidebar → **Download**, and place it into `D:\natural-presenter-ai\profiles\vn_01\lora\`.

In [ ]:
# === 1. Configuration & GPU Check ===
import os, sys, torch
from pathlib import Path

SPEAKER_ID = "vn_01"          # Change to "vn_02" or "ege_01" for Ege later
WORK_DIR = Path.cwd()          # Current JupyterLab folder
ZIP_PATH = WORK_DIR / f"{SPEAKER_ID}_f5_dataset.zip"

# L40 44GB optimized hyperparameters (~15-25 min total for ~27 min dataset)
EPOCHS = 100
LEARNING_RATE = 1e-5
BATCH_FRAMES = 6400            # 2x larger than L4 thanks to 44GB VRAM!
WARMUP_UPDATES = 100
SAVE_PER_UPDATES = 400
LAST_PER_UPDATES = 200

print(f"Working dir:  {WORK_DIR}")
print(f"Python:       {sys.version.split()[0]} | PyTorch: {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "CUDA GPU not detected!"
assert ZIP_PATH.exists(), f"Dataset zip not found at {ZIP_PATH}. Upload it via JupyterLab first!"
print(f"Dataset zip:  {ZIP_PATH.name} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB) — OK!")

In [ ]:
# === 2. Clone & Install F5-TTS (Preserving University Server's PyTorch CUDA) ===
import subprocess, sys
from pathlib import Path

F5_REPO = WORK_DIR / "F5-TTS"
if not F5_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/SWivid/F5-TTS.git", str(F5_REPO)], check=True)

# --upgrade-strategy only-if-needed keeps the server's working torch 2.6+cu124 untouched
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "-e", str(F5_REPO), "accelerate", "datasets", "soundfile"],
    check=True,
)

import f5_tts
print(f"F5-TTS ready at: {Path(f5_tts.__file__).resolve().parent}")

In [ ]:
# === 3. Extract Dataset, Build F5-TTS Arrow Format & Link Checkpoints ===
import csv, json, os, shutil, zipfile
from pathlib import Path
import soundfile as sf
from datasets.arrow_writer import ArrowWriter

RAW_DIR = WORK_DIR / "datasets" / SPEAKER_ID
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_DIR)

# Build F5-TTS data directory inside F5-TTS/data/{SPEAKER_ID}_pinyin
# Uses the exact 2545-token pretrained vocab (English stays raw chars -> preserves non-native accent!)
F5_DATA_DIR = F5_REPO / "data" / f"{SPEAKER_ID}_pinyin"
F5_DATA_DIR.mkdir(parents=True, exist_ok=True)

rows = []
durations = []
with (RAW_DIR / "metadata.csv").open("r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")
    header = next(reader, None)
    for rel_wav, text in reader:
        abs_wav = (RAW_DIR / rel_wav.strip()).resolve()
        if not abs_wav.exists():
            continue
        dur = float(sf.info(str(abs_wav)).duration)
        rows.append({"audio_path": str(abs_wav), "text": text.strip(), "duration": dur})
        durations.append(dur)

assert len(rows) > 0, "No valid WAV rows found!"

with ArrowWriter(path=str(F5_DATA_DIR / "raw.arrow")) as writer:
    for r in rows:
        writer.write(r)
    writer.finalize()

(F5_DATA_DIR / "duration.json").write_text(json.dumps({"duration": durations}, indent=2), encoding="utf-8")

pretrained_vocab = F5_REPO / "src" / "f5_tts" / "infer" / "examples" / "vocab.txt"
shutil.copy2(pretrained_vocab, F5_DATA_DIR / "vocab.txt")

# Symlink F5-TTS/ckpts/{SPEAKER_ID} -> WORK_DIR/ckpts/{SPEAKER_ID} for easy download in JupyterLab
LOCAL_CKPT_DIR = WORK_DIR / "ckpts" / SPEAKER_ID
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
F5_CKPT_LINK = F5_REPO / "ckpts" / SPEAKER_ID
F5_CKPT_LINK.parent.mkdir(parents=True, exist_ok=True)
if F5_CKPT_LINK.is_symlink() or F5_CKPT_LINK.exists():
    if F5_CKPT_LINK.is_symlink():
        F5_CKPT_LINK.unlink()
    else:
        shutil.rmtree(F5_CKPT_LINK)
os.symlink(str(LOCAL_CKPT_DIR.resolve()), str(F5_CKPT_LINK))

print(f"Dataset ready:   {len(rows)} clips | {sum(durations) / 60:.2f} minutes")
print(f"Arrow path:      {F5_DATA_DIR / 'raw.arrow'}")
print(f"Vocab size:      {len((F5_DATA_DIR / 'vocab.txt').read_text(encoding='utf-8').splitlines())} tokens")
print(f"Checkpoints dir: {LOCAL_CKPT_DIR}")

In [ ]:
# === 4. Launch F5-TTS Fine-Tuning on NVIDIA L40 (bf16, Full AdamW, 6400 Frames) ===
import subprocess, sys

cmd = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--mixed_precision=bf16",
    str(F5_REPO / "src" / "f5_tts" / "train" / "finetune_cli.py"),
    "--exp_name", "F5TTS_v1_Base",
    "--dataset_name", SPEAKER_ID,
    "--learning_rate", str(LEARNING_RATE),
    "--batch_size_per_gpu", str(BATCH_FRAMES),
    "--batch_size_type", "frame",
    "--epochs", str(EPOCHS),
    "--num_warmup_updates", str(WARMUP_UPDATES),
    "--save_per_updates", str(SAVE_PER_UPDATES),
    "--last_per_updates", str(LAST_PER_UPDATES),
    "--keep_last_n_checkpoints", "4",
    "--finetune",
    "--tokenizer", "pinyin",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# === 5. Listen & Compare: Zero-Shot Base vs. Your Fine-Tuned Checkpoints ===
from pathlib import Path
import soundfile as sf
from IPython.display import Audio, display
from f5_tts.api import F5TTS

TEST_SCRIPT = (
    "Hello everyone, today I want to, uh, walk you through our experiment results. "
    "We tested two model architectures across eighty trials, and, like, we noticed a clear drop "
    "in baseline error after tuning the encoder resolution."
)

REF_WAV = str(RAW_DIR / "refs" / "presentation_ref.wav")
REF_TXT = (RAW_DIR / "refs" / "presentation_ref.txt").read_text(encoding="utf-8").strip()
EVAL_DIR = WORK_DIR / "eval_samples" / SPEAKER_ID
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("=== 1. Base F5TTS_v1_Base (Zero-Shot Baseline) ===")
base_tts = F5TTS(device="cuda")
wav_base, sr, _ = base_tts.infer(ref_file=REF_WAV, ref_text=REF_TXT, gen_text=TEST_SCRIPT, nfe_step=32)
sf.write(str(EVAL_DIR / "00_zero_shot_base.wav"), wav_base, sr)
display(Audio(wav_base, rate=sr))

ckpts = sorted(
    [p for p in LOCAL_CKPT_DIR.glob("model_*.pt") if not p.name.startswith("pretrained_")]
    + [p for p in LOCAL_CKPT_DIR.glob("model_*.safetensors") if not p.name.startswith("pretrained_")]
)
print(f"\nFound {len(ckpts)} fine-tuned checkpoint(s) in {LOCAL_CKPT_DIR}:")
for ckpt in ckpts:
    print(f"\n=== Fine-Tuned: {ckpt.name} ({ckpt.stat().st_size / 1e6:.1f} MB) ===")
    ft_tts = F5TTS(ckpt_file=str(ckpt), device="cuda")
    wav_ft, sr, _ = ft_tts.infer(ref_file=REF_WAV, ref_text=REF_TXT, gen_text=TEST_SCRIPT, nfe_step=32)
    sf.write(str(EVAL_DIR / f"{ckpt.stem}.wav"), wav_ft, sr)
    display(Audio(wav_ft, rate=sr))

print(
    "\nDone! Right-click your favorite checkpoint in JupyterLab's left sidebar under "
    f"ckpts/{SPEAKER_ID}/ -> Download, then drop it into:\n"
    f"  D:\\natural-presenter-ai\\profiles\\{SPEAKER_ID}\\lora\\"
)